# Notebook 10: Combining Datasets
**Filename:** `10_Merging_Data.ipynb`  
**Topics Covered:** `concat()`, `merge()`, `join()`, `append()` (deprecation & alternatives), `combine_first()`

---

## 1. Concatenating Datasets (`concat`)

### Concept Explanation
`pd.concat()` glues or stacks DataFrames along a particular axis—either vertically along rows (`axis=0`) or horizontally along columns (`axis=1`). Parameters like `ignore_index=True` reset the index after combining.

### Real-world Example
Combining separate daily weather CSV logs from January and February into a single year-to-date dataset.

### Business Example
Merging quarterly sales spreadsheets with identical column headers from four distinct regional divisions.

### AI/ML Example
Stacking feature matrices ($X_1, X_2$) collected across different batch runs before running preprocessing pipelines.

In [2]:
import pandas as pd

df1 = pd.DataFrame({'ID': [1, 2], 'Sales': [100, 200]})
df2 = pd.DataFrame({'ID': [3, 4], 'Sales': [300, 400]})

# Vertical concatenation along rows (axis=0)
df_stacked = pd.concat([df1, df2], ignore_index=True)

# Horizontal concatenation along columns (axis=1)
df_side = pd.concat([df1, df2], axis=1)

print("Vertically Concatenated:\n", df_stacked)
print("\nHorizontally Concatenated:\n", df_side)


Vertically Concatenated:
    ID  Sales
0   1    100
1   2    200
2   3    300
3   4    400

Horizontally Concatenated:
    ID  Sales  ID  Sales
0   1    100   3    300
1   2    200   4    400


---

## 2. Database-Style Merging (`merge`)

### Concept Explanation
`pd.merge()` combines DataFrames based on common key columns using relational algebra rules (`how='inner'`, `'left'`, `'right'`, or `'outer'`). Keys can be specified using `on`, `left_on`, and `right_on`.

### Real-world Example
Merging water level readings with river gauge metadata matching on `Station_ID`.

### Business Example
Performing a left join between a `Customers` table and an `Orders` table to retain all registered users while attaching order records where available.

### AI/ML Example
Joining demographic customer features with historical transaction records to create unified model training matrices.

In [3]:
import pandas as pd

customers = pd.DataFrame({'Cust_ID': [101, 102, 103], 'Name': ['Alice', 'Bob', 'Charlie']})
orders = pd.DataFrame({'Cust_ID': [101, 102, 104], 'Amount': [250, 450, 150]})

# Inner Join (only matching IDs)
df_inner = pd.merge(customers, orders, on='Cust_ID', how='inner')

# Left Join (keep all customers)
df_left = pd.merge(customers, orders, on='Cust_ID', how='left')

# Outer Join (keep all records from both)
df_outer = pd.merge(customers, orders, on='Cust_ID', how='outer')

print("Inner Join:\n", df_inner)
print("\nLeft Join:\n", df_left)
print("\nOuter Join:\n", df_outer)

Inner Join:
    Cust_ID   Name  Amount
0      101  Alice     250
1      102    Bob     450

Left Join:
    Cust_ID     Name  Amount
0      101    Alice   250.0
1      102      Bob   450.0
2      103  Charlie     NaN

Outer Join:
    Cust_ID     Name  Amount
0      101    Alice   250.0
1      102      Bob   450.0
2      103  Charlie     NaN
3      104      NaN   150.0


---

## 3. Index-Based Joining (`join`)

### Concept Explanation
`df.join()` combines two DataFrames based primarily on row indices or a key column against an index. It defaults to a left outer join (`how='left'`).

### Real-world Example
Aligning time-series temperature data indexed by timestamp across two distinct weather stations.

### Business Example
Joining financial metrics indexed by `Store_ID` with operational store details.

### AI/ML Example
Attaching calculated feature vector DataFrames to original sample matrices sharing the same row index.

In [4]:
import pandas as pd

# DataFrames indexed by Store_ID
info = pd.DataFrame({'City': ['NY', 'LA']}, index=[101, 102])
revenue = pd.DataFrame({'Revenue': [50000, 75000]}, index=[101, 102])

# Join using index
df_joined = info.join(revenue)

print("Index Joined DataFrame:\n", df_joined)

Index Joined DataFrame:
     City  Revenue
101   NY    50000
102   LA    75000


---

## 4. `append()` Deprecation & Modern Alternatives

### Concept Explanation
The `df.append()` method was officially deprecated in Pandas version 1.4.0 and removed in 2.0. It was inefficient because it created a full copy of the DataFrame for every append operation ($O(N^2)$ time complexity).

### Modern Alternative
Use `pd.concat([df1, df2])` or collect intermediate dictionaries/DataFrames in a Python list and pass the entire list to `pd.concat()` once.

### Real-world Example
Collecting sensor telemetry batches in a list and concatenating them into a single master DataFrame at the end of execution.

### Business Example
Iterating over dynamic file sources and combining them efficiently using list concatenation instead of iterative appending.

### AI/ML Example
Accumulating batch inference results in memory and performing one global `pd.concat()` call.

In [5]:
import pandas as pd

# DEPRECATED (Do NOT use in modern Pandas):
# df_result = df1.append(df2)

# MODERN EFFICIENT ALTERNATIVE:
batch_list = []
batch_list.append(pd.DataFrame({'ID': [1], 'Val': [10]}))
batch_list.append(pd.DataFrame({'ID': [2], 'Val': [20]}))
batch_list.append(pd.DataFrame({'ID': [3], 'Val': [30]}))

# Combine all accumulated batches in a single operation
df_combined = pd.concat(batch_list, ignore_index=True)

print("Efficiently Combined DataFrame:\n", df_combined)

Efficiently Combined DataFrame:
    ID  Val
0   1   10
1   2   20
2   3   30


---

## 5. Combining Overlapping Datasets (`combine_first`)

### Concept Explanation
`combine_first()` fills null (`NaN`) values in one DataFrame using non-null values from another DataFrame at matching index and column positions.

### Real-world Example
Updating missing station rainfall values in a primary dataset using measurements from a secondary backup weather station.

### Business Example
Patching incomplete customer profile records with data retrieved from a secondary CRM backup database.

### AI/ML Example
Filling missing values in primary dataset feature matrices using imputations derived from secondary lookup tables.

In [6]:
import pandas as pd
import numpy as np

# Primary dataset with missing values
df_primary = pd.DataFrame({
    'Name': ['Alice', 'Bob', np.nan],
    'Salary': [70000, np.nan, 50000]
}, index=[101, 102, 103])

# Backup dataset with complete information
df_backup = pd.DataFrame({
    'Name': ['Alice', 'Bob', 'Charlie'],
    'Salary': [65000, 60000, 50000]
}, index=[101, 102, 103])

# Patch missing values in df_primary using df_backup
df_patched = df_primary.combine_first(df_backup)

print("Patched DataFrame:\n", df_patched)

Patched DataFrame:
         Name   Salary
101    Alice  70000.0
102      Bob  60000.0
103  Charlie  50000.0


---

## Minimum 5 Interview Questions with Answers

1. **What is the difference between `pd.concat()` and `pd.merge()`?**  
   * **Answer:** `pd.concat()` stacks or glues DataFrames along rows (`axis=0`) or columns (`axis=1`) based on indices or structural alignment. `pd.merge()` performs relational database joins based on shared key columns (`on`, `left_on`, `right_on`).

2. **Why was `df.append()` deprecated in Pandas, and what should be used instead?**  
   * **Answer:** `df.append()` created a complete copy of the underlying array upon every call, causing memory churn and $O(N^2)$ computational complexity. `pd.concat([list_of_dfs])` is the modern, high-performance replacement.

3. **What happens during a `left` join vs an `outer` join in `pd.merge()`?**  
   * **Answer:** A `left` join retains all rows from the left DataFrame and fills `NaN` for non-matching right DataFrame rows. An `outer` join retains all rows from both DataFrames, filling `NaN` wherever keys do not match.

4. **How does `df.join()` differ from `pd.merge()` by default?**  
   * **Answer:** `df.join()` joins DataFrames on their row indices by default using a left outer join (`how='left'`). `pd.merge()` joins on common column names by default using an inner join (`how='inner'`).

5. **When would you use `combine_first()` over `fillna()`?**  
   * **Answer:** Use `combine_first()` when you need to fill missing values across matching indices and columns using an entire secondary DataFrame rather than filling missing values with a scalar or constant column series.

---

## Self Reflection
* **What I Learned:** Mastered structural stacking (`concat`), relational merges (`merge`), index joins (`join`), modern list aggregation alternatives to `append()`, and patching overlapping datasets (`combine_first`).
* **Key Takeaway:** Combining datasets correctly is critical for building production data pipelines and creating unified feature tables from disparate sources.